In [1]:
import dotenv

dotenv.load_dotenv()

import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv("LOGFIRE_READ_TOKEN")
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [2]:
trace_rows = logfire_query_client.query_json_rows(sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 2
    """)

In [3]:
trace_ids = [r["trace_id"] for r in trace_rows["rows"]]
trace_ids

['019e27d0ea8b7aa56a24c3843563bbec', '019e27d0ea8b7aa56a24c3843563bbec']

In [4]:
trace_id = trace_ids[0]

run_row = logfire_query_client.query_json_rows(sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """)

In [5]:
all_messages = run_row["rows"][0]
all_messages["all_messages"][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': 'how do I determine which judge is wrong if they disagree?'}]}

In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import trace_replay

trace = trace_replay.fetch_trace(trace_id, logfire_query_client)
run = trace_replay.trace_to_run_result(trace)
run.output

{'answer': "To determine which LLM judge is wrong when there is disagreement between judges, you can implement the following steps:\n\n1. **Establish a Ground Truth**: Use a set of manual labels as a reference. This involves having a dataset where each entry has been vetted by a human evaluator to establish correctness. This will serve as the baseline to compare against the LLM judges' outputs.\n\n2. **Use a Reference-based Evaluation**: Set up your LLM evaluator to compare judge responses against the established ground truth. This way, you can identify whether the LLM judges are marking responses as correct or incorrect based on discrepancies with this ground truth.\n\n3. **Create Evaluation Descriptors**: Utilize the BinaryClassificationPromptTemplate to define the criteria for what constitutes a correct versus an incorrect response based on the reference answers. The template should be strict in its comparison to ensure that even subtle discrepancies are captured.\n\n4. **Run Evalua

In [10]:
traces = trace_replay.fetch_traces(trace_ids, logfire_query_client)

runs = []

for trace in traces.values():
    run = trace_replay.trace_to_run_result(trace)
    runs.append(run)

In [13]:
from pathlib import Path
import pickle

output_dir = Path("../data")

output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "logs.bin", "wb") as f_out:
    pickle.dump(runs, f_out)

In [18]:
from pathlib import Path

import pickle

with open(Path("../data/logs.bin"), "rb") as f_in:
    runs = pickle.load(f_in)